✅ 4. ParentDocumentRetriever

Use Case: Retrieve parent chunks based on a retrieval of smaller child chunks, great for keeping context intact in long documents.

In [2]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.storage import InMemoryStore
from langchain_core.documents import Document
from dotenv import load_dotenv
import os

load_dotenv()

# 1. Setup Gemini embeddings
embedding = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001",
    google_api_key=os.getenv("GOOGLE_API_KEY")
)

# 2. Prepare documents
docs = [
    Document(page_content="LangChain helps build LLM-powered apps with memory and agents.", metadata={"id": "1"}),
    Document(page_content="Agents in LangChain use tools to answer questions.", metadata={"id": "2"})
]

# 3. Setup child splitter
child_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)

# 4. Setup vectorstore
vectorstore = FAISS.from_texts(["placeholder"], embedding)
vectorstore.delete([vectorstore.index_to_docstore_id[0]])

# 5. Parent retriever
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=InMemoryStore(),
    child_splitter=child_splitter
)

# 6. Add documents
retriever.add_documents(docs)

# 7. Retrieve
results = retriever.invoke("What are agents?")
for doc in results:
    print("📄 Retrieved Doc:", doc.page_content)

📄 Retrieved Doc: Agents in LangChain use tools to answer questions.
📄 Retrieved Doc: LangChain helps build LLM-powered apps with memory and agents.


In [ ]:
![image.png](attachment:b7f0fa9f-a784-41ba-88fd-2d02c17c19b4.png)

In [5]:
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document

# Create simple text docs
docs = [
    Document(page_content="LangChain enables LLM applications."),
    Document(page_content="Vector search is powerful."),
    Document(page_content="BM25 is a classical retrieval method.")
]

# Create BM25 retriever
bm25_retriever = BM25Retriever.from_documents(docs)

# Retrieve — use invoke(), get_relevant_documents is deprecated
results = bm25_retriever.invoke("How does BM25 work?")
for doc in results:
    print("📝 BM25 Result:", doc.page_content)

📝 BM25 Result: BM25 is a classical retrieval method.
📝 BM25 Result: Vector search is powerful.
📝 BM25 Result: LangChain enables LLM applications.


In [4]:
#!pip install rank_bm25

In [7]:
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.documents import Document
from dotenv import load_dotenv
import os

load_dotenv()

# Sample docs
docs = [
    Document(page_content="LangChain supports LLMs."),
    Document(page_content="You can build AI apps using LangChain.")
]

# BM25 Retriever (keyword-based, no embeddings/API key needed)
bm25 = BM25Retriever.from_documents(docs)

# Vector Retriever (Gemini embeddings)
embedding = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001",
    google_api_key=os.getenv("GOOGLE_API_KEY")
)
vectorstore = FAISS.from_documents(docs, embedding)
vector_retriever = vectorstore.as_retriever()

# Ensemble Retriever (equal weight)
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25, vector_retriever],
    weights=[0.5, 0.5]
)

# Query
results = ensemble_retriever.invoke("AI apps using LangChain")
for doc in results:
    print("🔍 Ensemble Doc:", doc.page_content)

🔍 Ensemble Doc: You can build AI apps using LangChain.
🔍 Ensemble Doc: LangChain supports LLMs.


7.TimeWeightedVectorStoreRetriever

This retriever boosts document relevance by factoring recency and importance of interactions (used in agents with memory or relevance ranking).

In [8]:
![image.png](attachment:8a556f26-c107-4068-b36e-6a567bb35088.png)



'[image.png]' is not recognized as an internal or external command,
operable program or batch file.


In [9]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_classic.retrievers import TimeWeightedVectorStoreRetriever
from langchain_core.documents import Document
from datetime import datetime
from dotenv import load_dotenv
import os

load_dotenv()

# Initialize embedding & vectorstore
embedding = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001",
    google_api_key=os.getenv("GOOGLE_API_KEY")
)

docs = [
    Document(page_content="LangChain is for LLM-based apps", metadata={"last_accessed_at": datetime.now()}),
    Document(page_content="Vector search improves relevance", metadata={"last_accessed_at": datetime.now()})
]
vectorstore = FAISS.from_documents(docs, embedding)

# TimeWeighted Retriever
retriever = TimeWeightedVectorStoreRetriever(
    vectorstore=vectorstore,
    decay_rate=0.01,
    k=2,
    score_threshold=None
)

# Retrieve
results = retriever.invoke("What is LangChain?")
for r in results:
    print(r)
    print(r.page_content)

In [12]:
from langchain_community.retrievers import TavilySearchAPIRetriever
from dotenv import load_dotenv
import os

# 🔐 Load API keys
load_dotenv(".env")
google_api_key = os.getenv("GOOGLE_API_KEY")
tavily_api_key = os.getenv("TAVILY_API_KEY")

# Tavily reads TAVILY_API_KEY from the environment automatically,
# but you can also confirm it's loaded:
if not tavily_api_key:
    raise ValueError("TAVILY_API_KEY not found — check your .env file")

# Initialize Tavily Retriever
retriever = TavilySearchAPIRetriever(k=3)

# Perform search — use invoke(), get_relevant_documents is deprecated
docs = retriever.invoke("Latest updates about LangChain")
for doc in docs:
    print(doc.page_content)

Jul 23, 2026
  + First seen by Releasebot:

    Jul 24, 2026

  LangChain logo

  LangChain

  LangChain adds LangSmith Gateway env var support and fixes the gpt-5.3-chat-latest profile.

  ### Changes since langchain-openai==1.4.0

  release(openai): 1.4.1 (#39045)

  feat(anthropic,fireworks,openai): support langsmith gateway through env var (#38742)

  fix(openai): correct gpt-5.3-chat-latest profile (#39009)

  Original source
 Jul 23, 2026 
  + Date parsed from source: [...] Jul 23, 2026
  + First seen by Releasebot:

    Jul 24, 2026

  LangChain logo

  LangChain

  LangChain adds LangSmith Gateway support through an environment variable for Anthropic, Fireworks, and OpenAI.

  ### Changes since langchain-fireworks==1.5.0

  release(fireworks): 1.5.1 (#39044)

  feat(anthropic,fireworks,openai): support langsmith gateway through env var (#38742)

  Original source

Releasebot

## Curated by the Releasebot team [...] LangChain releases langchain-openai 1.4.2 with OpenAI fixes for

In [11]:
#!pip install tavily-python